In [7]:
from astropy.io import fits
from astropy.visualization import simple_norm
from astropy.visualization import SqrtStretch, AsinhStretch, ImageNormalize, ZScaleInterval
from astropy.visualization.stretch import SinhStretch, LinearStretch
import math
import numpy as np


import datetime
import numpy as np
import asdf
from asdf.tags.core import Software, HistoryEntry
from astropy import units as u
from jwst.datamodels import ImageModel
from jwst.datamodels import WfssBkgModel
from astropy.io import fits
from stdatamodels.jwst.datamodels import dqflags
from stdatamodels.jwst import datamodels

In [10]:
# Number the flat DQ plane 
flat_file = "jwst_miri_flat_0844.fits"
flat_output = "jwst_miri_flat_WFSS.fits"



In [30]:
hdu = fits.open(flat_file)
model = datamodels.FlatModel(flat_file)
data = model.data
dq  = model.dq
err = model.err

new_model = model.copy()

In [20]:
#-------------------------------------------------------------------------------
def create_dqdef():
    """Create the DQ definition data needed to describe the DQ plane.

    Returns
    -------
    definitions : list
        Bad pixel bit definitions
    """
    definitions = []
    standard_defs = dqflags.pixel
    for bitname in standard_defs:
        bitvalue = standard_defs[bitname]
        if bitvalue != 0:
            bitnumber = np.uint8(np.log(bitvalue)/np.log(2))
        else:
            bitnumber = 0
        newrow = (bitnumber, bitvalue, bitname, '')
        definitions.append(newrow)
    return definitions

In [35]:



def create_miri_wfss_bkg(new_model,
                         flat_output,
                         author="Andreea Petric",
                         history=None,
                         pedigree="INFLIGHT 2022-05-22 2022-05-22",
                         useafter="2022-01-01T00:00:00"):
    """
    Create the MIRI WFSS background reference file
    Parameters
    ----------
    datafile : str
        The text file containing the bkg data.
    filter : str
        P750L
    outname : str
        Output name for the reference file.
    author : str
        The name of the author.
    history : str
        A comment about the refrence file to be saved with
        the meta information.
    pedigree : str
        GROUND or INFLIGHT with the dates of the data used to create the
        reference file.
    useafter : str
        Everything on this date or later will use this file (unless a file with
        an older useafter exists).


    Returns
    -------
    writes out the fits reference file
    """


    author="Andreea Petric"
    description="MIRI WFSS Flat Ref File"
    exp_type='MIR_WFSS'
    pedigree="INFLIGHT 2022-05-22 2024-08-04"
    reftype='FLAT'
    filter_type = 'P750L'
    title="MIRI Reference File"
    useafter="2022-01-01T00:00:00"

    new_model.meta.author = author
    new_model.meta.description = description
    new_model.meta.telescope = "JWST"
    new_model.meta.useafter = useafter
    new_model.meta.title = title
    new_model.meta.pedigree = pedigree
    new_model.meta.reftype = reftype
    new_model.meta.filter = filter_type

    # 'meta' field:
    new_model.meta.input_units = u.micron
    new_model.meta.output_units = u.micron

    new_model.meta.exposure.type = exp_type
    print(exp_type)

    new_model.meta.instrument.name = "MIRI"
    new_model.meta.instrument.detector="MIRIMAGE"
    

    new_model.meta.instrument.band = 'N/A'
    new_model.meta.instrument.channel ='N/A'
    new_model.meta.subarray.name = 'FULL'
    

    # flag the Lyot mask
    x1 = 4
    x2 = 275
    y1 = 752
    y2 = 1020
    new_model.data[y1:y2, x1:x2] = np.nan
    new_model.err[y1:y2, x1:x2] = np.nan
    new_model.dq[y1:y2, x1:x2] = 1
    
    history =" MIRI WFSS created from jwst_miri_flat_0844.fits"
    # history entries (also updates meta['history'] field)
    entry = HistoryEntry({'description': history,
                          'time': datetime.datetime.utcnow()})
    sdict = Software({'name': 'MIRI_WFSS_Flatr4ref.ipynb',
                      'author': author,
                      'version': '0.1.0'})
    entry['software'] = sdict
    new_model.meta.history = history
    new_model.history.append(entry)
    # writing & validating ref. file
    new_model.validate()
    print(new_model.meta)
    print(flat_output)
    new_model.save(flat_output)





In [36]:
create_miri_wfss_bkg(new_model,
                         flat_output,
                         author="Andreea Petric",
                         history=None,
                         pedigree="INFLIGHT 2022-05-22 2022-05-22",
                         useafter="2022-01-01T00:00:00")


MIR_WFSS
jwst_miri_flat_WFSS.fits


/var/folders/l0/s51kjpx95j12hwymx_lfmg6h000105/T/ipykernel_73134/1935689500.py:82: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'time': datetime.datetime.utcnow()})
/var/folders/l0/s51kjpx95j12hwymx_lfmg6h000105/T/ipykernel_73134/1935689500.py:88: UserWarning: The history attribute will soon be deprecated. Use add_history_entry to add history entries
  new_model.history.append(entry)
